# Synthetic Generation (Gemini Flash + .env)

This notebook runs on Colab CPU or GPU (no local model loading).
1. Load `GEMINI_KEY` from `.env` (or Colab secrets fallback).
2. Read parsed terms from Drive (`/content/drive/MyDrive/dap-hs/singlish_parsed.jsonl`).
3. Generate synthetic outputs with Gemini Flash to Drive (`synthetic_training_gemini.jsonl`).

In [1]:
# Install dependencies (new Gemini SDK)
%pip install -q google-genai python-dotenv requests

## Configure Gemini API key
Looks for `GEMINI_KEY` in `.env` first, then falls back to Colab Secrets (`GEMINI_KEY` / `GOOGLE_API_KEY`).

In [17]:
from google import genai

# CANNOT COMMIT THIS NOTEBOOK
GEMINI_KEY = "REDACTED_GEMINI_KEY"
GEMINI_MODEL = "gemini-3-flash-preview"

if not GEMINI_KEY or GEMINI_KEY == "PASTE_YOUR_GEMINI_KEY_HERE":
    raise ValueError("Set GEMINI_KEY in this cell before running.")

GENAI_CLIENT = genai.Client(api_key=GEMINI_KEY)

print("✅ Gemini configured")
print("✅ Model:", GEMINI_MODEL)
print("✅ GEMINI_KEY loaded:", bool(GEMINI_KEY))

✅ Gemini configured
✅ Model: gemini-3-flash-preview
✅ GEMINI_KEY loaded: True


In [18]:
# Optional: list available models
for m in GENAI_CLIENT.models.list():
    name = getattr(m, 'name', '')
    if 'flash' in name.lower():
        print(name)

models/gemini-2.5-flash
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-exp-image-generation
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-2.5-flash-lite-preview-09-2025
models/gemini-3-flash-preview
models/gemini-3.1-flash-image-preview
models/gemini-2.5-flash-native-audio-latest
models/gemini-2.5-flash-native-audio-preview-09-2025
models/gemini-2.5-flash-native-audio-preview-12-2025


In [19]:
# Mount Google Drive + set a dedicated working folder for this project
USE_GOOGLE_DRIVE = True

from pathlib import Path

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        print('Google Drive mounted at /content/drive')
    except Exception as e:
        print('Drive mount skipped or unavailable:', e)

drive_root = Path('/content/drive/MyDrive')
DAP_HS_DIR = drive_root / 'dap-hs'
DAP_HS_DIR.mkdir(parents=True, exist_ok=True)

DRIVE_REPO_PATH = DAP_HS_DIR / '11th-DAP-hate-speech'
DRIVE_PARSED_FILE = DAP_HS_DIR / 'singlish_parsed.jsonl'
DRIVE_OUTPUT_FILE = DAP_HS_DIR / 'synthetic_training_gemini.jsonl'

if DRIVE_REPO_PATH.exists():
    PROJECT_ROOT = DRIVE_REPO_PATH
    print('Using PROJECT_ROOT from Drive repo:', PROJECT_ROOT)
else:
    PROJECT_ROOT = Path.cwd()
    print('No Drive repo folder found; using runtime PROJECT_ROOT:', PROJECT_ROOT)

print('DAP_HS_DIR:', DAP_HS_DIR)
print('DRIVE_PARSED_FILE:', DRIVE_PARSED_FILE, '| exists:', DRIVE_PARSED_FILE.exists())
print('DRIVE_OUTPUT_FILE:', DRIVE_OUTPUT_FILE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted at /content/drive
No Drive repo folder found; using runtime PROJECT_ROOT: /content
DAP_HS_DIR: /content/drive/MyDrive/dap-hs
DRIVE_PARSED_FILE: /content/drive/MyDrive/dap-hs/singlish_parsed.jsonl | exists: True
DRIVE_OUTPUT_FILE: /content/drive/MyDrive/dap-hs/synthetic_training_gemini.jsonl


In [20]:
# Upload parsed file if missing
if not DRIVE_PARSED_FILE.exists():
    print('DRIVE_PARSED_FILE not found, please upload singlish_parsed.jsonl...')
    from google.colab import files
    uploaded = files.upload()
    if uploaded:
        first_name = next(iter(uploaded.keys()))
        Path('singlish_parsed.jsonl').write_bytes(uploaded[first_name])
        DRIVE_PARSED_FILE.write_bytes(Path('singlish_parsed.jsonl').read_bytes())
        print('Saved to:', DRIVE_PARSED_FILE)
    else:
        print('No file uploaded.')
else:
    print('Found parsed file at:', DRIVE_PARSED_FILE)

Found parsed file at: /content/drive/MyDrive/dap-hs/singlish_parsed.jsonl


In [21]:
import json
import re
import time

SG_TARGET_GROUPS = {
    'chinese': 'Chinese Singaporeans (majority race, includes dialect groups: Hokkien, Teochew, Cantonese, Hakka, Hainanese)',
    'malay': 'Malay Singaporeans (includes Muslim Malays)',
    'indian': 'Indian Singaporeans (includes Tamil, Hindi, Sikh, Malayalee communities)',
    'others': 'Eurasians, mixed-race, and other minority Singaporeans',
    'migrant_worker': 'Foreign migrant workers (Bangladeshi, Chinese nationals/PRCs, Vietnamese, Filipino, Myanmar, Indonesian)',
    'religion': 'Religious groups (Islam, Christianity, Buddhism, Hinduism, Taoism, Sikhism)',
}

LANG_TO_COMMUNITY = {
    'hokkien': 'chinese', 'teochew': 'chinese', 'cantonese': 'chinese',
    'hakka': 'chinese', 'hainanese': 'chinese', 'mandarin': 'chinese',
    'min nan': 'chinese', 'chinese': 'chinese',
    'malay': 'malay', 'bahasa': 'malay', 'jawi': 'malay',
    'tamil': 'indian', 'hindi': 'indian', 'punjabi': 'indian',
    'sanskrit': 'indian', 'urdu': 'indian',
    'bengali': 'migrant_worker', 'tagalog': 'migrant_worker',
    'filipino': 'migrant_worker', 'vietnamese': 'migrant_worker',
    'burmese': 'migrant_worker', 'myanmar': 'migrant_worker',
    'arabic': 'religion',
}

PROTECTED_GROUP_KEYWORDS = [
    'chinese', 'malay', 'indian', 'eurasian',
    'migrant', 'worker', 'bangladeshi', 'bangla', 'filipino', 'vietnamese', 'myanmar', 'burmese', 'prc',
    'muslim', 'islam', 'christian', 'buddhist', 'hindu', 'sikh', 'religion',
    'tamil', 'hindi', 'punjabi',
    'foreigner', 'foreign worker',
]

RESPONSE_JSON_SCHEMA = {
    'type': 'array',
    'items': {
        'type': 'object',
        'properties': {
            'text': {'type': 'string'},
            'label': {'type': 'string', 'enum': ['hate', 'offensive', 'normal']},
            'target_group': {
                'type': 'string',
                'enum': ['none', 'chinese', 'malay', 'indian', 'others', 'migrant_worker', 'religion'],
            },
        },
        'required': ['text', 'label', 'target_group'],
        'additionalProperties': True,
    },
}

SYSTEM_PROMPT = """You generate Singapore social media posts for a hate speech detection research dataset.

You will be given a Singlish term and its meaning. Write posts that use that exact term.

Output ONLY a JSON array. No markdown, no explanation, no extra text.

Each object: {\"text\": \"...\", \"label\": \"hate|offensive|normal\", \"target_group\": \"...\"}

Label policy:
- hate: MUST target a protected group and explicitly mention that target group in the text
- offensive: rude/profane but does NOT target a protected group (target_group must be \"none\")
- normal: neutral/benign usage (target_group must be \"none\")

Fairness rule for hate samples:
- For each term, spread hate examples across ALL provided target groups (chinese, malay, indian, others, migrant_worker, religion) as evenly as possible.
- Do not focus hate examples on only one group.

Language rule (strict): Write in English only. You may use Singlish particles (lah, leh, lor, sia) and the given term, but do not write full Malay/Chinese/Tamil sentences."""

def infer_target_groups(languages):
    groups = {LANG_TO_COMMUNITY[l] for l in (languages or []) if l in LANG_TO_COMMUNITY}
    return list(groups) if groups else list(SG_TARGET_GROUPS.keys())

def extract_context_from_parsed(entry):
    definitions = (entry.get('definitions') or '').strip()
    usage_notes = (entry.get('usage_notes') or '').strip()
    synonyms = entry.get('synonyms') or []
    antonyms = entry.get('antonyms') or []

    parts = []
    if definitions:
        parts.append(f'Definitions:\n{definitions}')
    if usage_notes:
        parts.append(f'Usage notes:\n{usage_notes}')
    if synonyms:
        parts.append('Synonyms: ' + ', '.join(synonyms))
    if antonyms:
        parts.append('Antonyms: ' + ', '.join(antonyms))

    text = '\n\n'.join(parts).strip()
    return text[:1200] + ('\n...' if len(text) > 1200 else '')

def normalize_entry(entry):
    categories = entry.get('categories') or entry.get('source_categories') or []
    if isinstance(categories, str):
        categories = [categories]
    return {
        'term': (entry.get('term') or '').strip(),
        'categories': categories,
        'languages': entry.get('languages') or [],
        'context': extract_context_from_parsed(entry),
    }

def build_prompt(term, categories, context, n_per_label, target_groups, languages):
    cat_str = ', '.join(categories) if categories else 'general'
    lang_str = ', '.join(languages) if languages else 'mixed/unclear'

    unique_groups = [g for g in target_groups if g in SG_TARGET_GROUPS]
    if not unique_groups:
        unique_groups = list(SG_TARGET_GROUPS.keys())

    group_descriptions = [f'- {g}: {SG_TARGET_GROUPS[g]}' for g in unique_groups]
    group_block = '\n'.join(group_descriptions)

    hate_count = max(n_per_label, len(unique_groups))

    return f'''Write Singapore social media posts that all use the Singlish term "{term}".

The word/phrase "{term}" MUST be written in every single post. This is mandatory.

What "{term}" means:
{context}

Categories: {cat_str} | Language origins: {lang_str}

Target groups to cover for hate examples (must cover all):
{group_block}

Write exactly:
- {hate_count} "hate" posts (distributed across all target groups above; each group should appear at least once where possible)
- {n_per_label} "offensive" posts (rude but NOT targeting a protected group)
- {n_per_label} "normal" posts (neutral/benign usage)

Rules:
- Every post MUST contain "{term}".
- Every hate post MUST include explicit group mention in the text and set matching target_group.
- Offensive/normal posts MUST set target_group to "none".
- Plain text only. English Singlish only (English sentence structure; Singlish particles allowed).
- Keep each post 1-3 sentences.
- Do NOT output full Malay/Chinese/Tamil sentences.

JSON array only, no other text:
[
  {{"text": "... {term} ...", "label": "hate", "target_group": "malay"}},
  {{"text": "... {term} ...", "label": "offensive", "target_group": "none"}},
  {{"text": "... {term} ...", "label": "normal", "target_group": "none"}}
]'''

def gemini_chat_generate(system_prompt, user_prompt, temperature=0.2):
    combined_prompt = f'SYSTEM INSTRUCTION:\n{system_prompt}\n\nUSER REQUEST:\n{user_prompt}'
    response = GENAI_CLIENT.models.generate_content(
        model=GEMINI_MODEL,
        contents=combined_prompt,
        config={
            'temperature': temperature,
            'top_p': 0.9,
            'max_output_tokens': 2048,
            'response_mime_type': 'application/json',
            'response_json_schema': RESPONSE_JSON_SCHEMA,
        },
    )
    text = getattr(response, 'text', None)

    finish_reasons = []
    for cand in getattr(response, 'candidates', []) or []:
        reason = getattr(cand, 'finish_reason', None)
        if reason is not None:
            finish_reasons.append(str(reason))

    if text:
        return text.strip(), {'finish_reasons': finish_reasons}

    chunks = []
    for cand in getattr(response, 'candidates', []) or []:
        content = getattr(cand, 'content', None)
        parts = getattr(content, 'parts', None) if content else None
        if parts:
            for p in parts:
                t = getattr(p, 'text', None)
                if t:
                    chunks.append(t)

    return '\n'.join(chunks).strip(), {'finish_reasons': finish_reasons}

def parse_llm_output(raw_text):
    text = (raw_text or '').strip()
    text = re.sub(r'^```(?:json)?\s*', '', text)
    text = re.sub(r'\s*```$', '', text)

    try:
        data = json.loads(text)
        if isinstance(data, dict):
            if 'text' in data and 'label' in data:
                data = [data]
            else:
                for v in data.values():
                    if isinstance(v, list):
                        data = v
                        break
                else:
                    return []
        if not isinstance(data, list):
            return []
    except json.JSONDecodeError:
        match = re.search(r'\[.*\]', text, re.DOTALL)
        if not match:
            return []
        try:
            data = json.loads(match.group())
        except json.JSONDecodeError:
            return []

    valid = []
    for item in data:
        if isinstance(item, dict) and 'text' in item and 'label' in item:
            label = str(item['label']).strip().lower()
            if label in ('hate', 'offensive', 'normal'):
                if 'target_group' not in item:
                    item['target_group'] = 'none'
                item['label'] = label
                valid.append(item)
    return valid

def term_appears_in_text(term, text):
    text_lower = (text or '').lower()
    term_lower = term.lower().strip()

    if term_lower in text_lower:
        return True

    clean_term = term_lower.replace('…', '').replace('...', '')
    core_words = [w for w in clean_term.split() if len(w) > 1]
    if core_words:
        matches = sum(1 for w in core_words if w in text_lower)
        if matches == len(core_words):
            return True

    stop_words = {'buay', 'the', 'and', 'for', 'a', 'is', 'to', 'of', 'in'}
    words = [w for w in term_lower.split() if len(w) > 2 and w not in stop_words and '.' not in w and '…' not in w]
    if words:
        matches = sum(1 for w in words if w in text_lower)
        if matches >= max(1, len(words) * 0.6):
            return True

    if term.isupper() and len(term) <= 6 and term in (text or ''):
        return True

    return False

def text_has_protected_group(text):
    text_lower = (text or '').lower()
    return any(k in text_lower for k in PROTECTED_GROUP_KEYWORDS)

def validate_examples(term, examples):
    accepted = []
    reject_reasons = []
    allowed_groups = set(SG_TARGET_GROUPS.keys())

    for ex in examples:
        label = str(ex.get('label', '')).lower().strip()
        tg = str(ex.get('target_group', 'none')).lower().strip()
        text = ex.get('text', '')

        if not term_appears_in_text(term, text):
            reject_reasons.append('missing_source_term')
            continue

        if label == 'hate':
            if tg in ('', 'none') or tg not in allowed_groups:
                reject_reasons.append('hate_invalid_target_group')
                continue
            if not text_has_protected_group(text):
                reject_reasons.append('hate_missing_protected_group_in_text')
                continue
        else:
            ex['target_group'] = 'none'

        accepted.append(ex)

    return accepted, reject_reasons

def _safe_preview(text, max_len=500):
    if not text:
        return ''
    text = str(text).replace('\n', '\\n')
    return text[:max_len] + ('...' if len(text) > max_len else '')

def preview_prompts(input_file, selected_terms, n_per_label=2, limit=3):
    entries = [normalize_entry(json.loads(line)) for line in Path(input_file).open(encoding='utf-8') if line.strip()]
    entries = [e for e in entries if e['term']]
    if selected_terms:
        emap = {e['term'].casefold(): e for e in entries}
        entries = [emap[t.casefold()] for t in selected_terms if t.casefold() in emap]

    for entry in entries[:limit]:
        groups = infer_target_groups(entry['languages'])
        prompt = build_prompt(entry['term'], entry['categories'], entry['context'], n_per_label, groups, entry['languages'])
        print('\n' + '=' * 80)
        print('TERM:', entry['term'])
        print('=' * 80)
        print(prompt)

def generate_dataset(input_file, output_file, selected_terms, n_per_label=2, limit=None, temperature=0.2, max_retries=3):
    input_file = Path(input_file)
    output_file = Path(output_file)
    output_file.parent.mkdir(parents=True, exist_ok=True)
    failure_log_file = output_file.with_suffix(output_file.suffix + '.failures.log')

    entries = [normalize_entry(json.loads(line)) for line in input_file.open(encoding='utf-8') if line.strip()]
    entries = [e for e in entries if e['term']]

    if selected_terms:
        emap = {e['term'].casefold(): e for e in entries}
        entries = [emap[t.casefold()] for t in selected_terms if t.casefold() in emap]

    if limit:
        entries = entries[:limit]

    total_written = 0
    with output_file.open('a', encoding='utf-8') as fh, failure_log_file.open('a', encoding='utf-8') as flog:
        for idx, entry in enumerate(entries, start=1):
            term = entry['term']
            if not entry['context']:
                print(f'[{idx}/{len(entries)}] {term} ... SKIP (no context)')
                continue

            groups = infer_target_groups(entry['languages'])
            prompt = build_prompt(term, entry['categories'], entry['context'], n_per_label, groups, entry['languages'])
            print(f'[{idx}/{len(entries)}] {term} ...', end=' ', flush=True)

            final_examples = []
            attempt_logs = []

            for attempt in range(1, max_retries + 1):
                raw, meta = gemini_chat_generate(SYSTEM_PROMPT, prompt, temperature=temperature)
                finish_reasons = (meta or {}).get('finish_reasons') or []

                attempt_info = {
                    'attempt': attempt,
                    'finish_reasons': finish_reasons,
                    'raw_preview': _safe_preview(raw),
                    'reason': None,
                }

                if not raw:
                    reason = 'empty_response'
                    if finish_reasons:
                        reason += f'|finish_reasons={finish_reasons}'
                    attempt_info['reason'] = reason
                    attempt_logs.append(attempt_info)
                    continue

                parsed = parse_llm_output(raw)
                if not parsed:
                    attempt_info['reason'] = 'invalid_or_non_json_output'
                    attempt_logs.append(attempt_info)
                    continue

                parsed = [e for e in parsed if term_appears_in_text(term, e.get('text', ''))]
                if not parsed:
                    attempt_info['reason'] = 'missing_source_term_after_parse'
                    attempt_logs.append(attempt_info)
                    continue

                accepted, rejected = validate_examples(term, parsed)
                if not accepted:
                    attempt_info['reason'] = 'validation_failed:' + ','.join(rejected[:5])
                    attempt_logs.append(attempt_info)
                    continue

                attempt_info['reason'] = 'success'
                attempt_logs.append(attempt_info)
                final_examples = accepted
                if attempt > 1:
                    print(f'recovered on retry {attempt}', end='; ', flush=True)
                break

            if not final_examples:
                print('FAILED')
                flog.write(json.dumps({
                    'term': term,
                    'source_category': (entry['categories'][0] if entry['categories'] else ''),
                    'attempt_logs': attempt_logs,
                    'timestamp': time.time(),
                }, ensure_ascii=False) + '\n')
                if attempt_logs:
                    print('  fail_reason:', attempt_logs[-1].get('reason'))
                    print('  raw_preview:', attempt_logs[-1].get('raw_preview', '')[:180])
                continue

            for ex in final_examples:
                fh.write(json.dumps({
                    'text': ex['text'],
                    'label': ex['label'],
                    'target_group': ex.get('target_group', 'none'),
                    'source_term': term,
                    'source_category': (entry['categories'][0] if entry['categories'] else ''),
                }, ensure_ascii=False) + '\n')
                total_written += 1

            print(f'OK ({len(final_examples)} rows)')
            time.sleep(0.4)

    print('Done. Wrote', total_written, 'rows to', output_file)
    print('Failure log:', failure_log_file)

## Configure generation

In [25]:
INPUT_FILE = DRIVE_PARSED_FILE
OUTPUT_FILE = DRIVE_OUTPUT_FILE

PER_TERM = 6
LIMIT = None
TEMPERATURE = 0.2

SELECTED_TERMS_RAW = '''
anak babi
apu neh neh
chao chee bai
diu lei lo mo
kan ni na bu
NBCB
pariah
pubor kia
pukimak
siao lang
tu lan
'''

SELECTED_TERMS = [t.strip() for t in re.split(r'[,\n;]', SELECTED_TERMS_RAW) if t.strip()]

print('Model:', GEMINI_MODEL)
print('Input:', INPUT_FILE)
print('Output:', OUTPUT_FILE)
print('Terms:', len(SELECTED_TERMS), SELECTED_TERMS)

Model: gemini-3-flash-preview
Input: /content/drive/MyDrive/dap-hs/singlish_parsed.jsonl
Output: /content/drive/MyDrive/dap-hs/synthetic_training_gemini.jsonl
Terms: 11 ['anak babi', 'apu neh neh', 'chao chee bai', 'diu lei lo mo', 'kan ni na bu', 'NBCB', 'pariah', 'pubor kia', 'pukimak', 'siao lang', 'tu lan']


## Reset previous generated files
Optional: clear old outputs before regenerating to avoid mixing runs.

In [23]:
RESET_OUTPUT_FILES = True  # Set False if you want to append instead

if RESET_OUTPUT_FILES:
    output_path = Path(OUTPUT_FILE)
    failure_log_path = output_path.with_suffix(output_path.suffix + '.failures.log')
    cleanup_backup_path = output_path.with_suffix(output_path.suffix + '.backup_before_group_cleanup')

    for file_path in [output_path, failure_log_path, cleanup_backup_path]:
        if file_path.exists():
            file_path.unlink()
            print('Deleted:', file_path)
        else:
            print('Not found (skip):', file_path)
else:
    print('RESET_OUTPUT_FILES is False; keeping existing output files.')

Deleted: /content/drive/MyDrive/dap-hs/synthetic_training_gemini.jsonl
Not found (skip): /content/drive/MyDrive/dap-hs/synthetic_training_gemini.jsonl.failures.log
Not found (skip): /content/drive/MyDrive/dap-hs/synthetic_training_gemini.jsonl.backup_before_group_cleanup


## Dry run (preview prompt text)

In [ ]:
preview_prompts(
    input_file=INPUT_FILE,
    selected_terms=SELECTED_TERMS,
    n_per_label=PER_TERM,
    limit=3,
)


TERM: anak babi
Write Singapore social media posts that all use the Singlish term "anak babi".

The word/phrase "anak babi" MUST be written in every single post. This is mandatory.

What "anak babi" means:
Definitions:
1.   (pejorative) 'pig child'; 'child of a pig'; term of disdain, contempt, displeasure, anger, etc. especially used to scold a child

Court documents state that Madam Marliah was cleaning her home on April 24 last year when Lee became unhappy with the noise caused by her vacuuming. From her flat, she shouted the words "anak babi", which means "baby pig" in Malay, at Madam Marliah.

— 2020 Feb 6, Dominic Low. The Straits Times, "12 months' probation for woman who wounded neighbour's religious feelings".

Usage notes:
Particularly offensive to Muslims, due to the association of pigs with impurity in Islam.

Categories: insults | Language origins: mixed/unclear

Target groups to cover for hate examples (must cover all):
- chinese: Chinese Singaporeans (majority race, incl

## Targeted per-group generation mode
Generate smaller structured responses per target group/label to reduce truncation and keep partial results.

In [32]:
GROUP_KEYWORDS = {
    'chinese': ['chinese', 'hokkien', 'teochew', 'cantonese', 'mandarin', 'hakka', 'hainanese'],
    'malay': ['malay'],
    'indian': ['indian', 'tamil', 'hindi', 'punjabi', 'sikh'],
    'others': ['eurasian', 'mixed-race', 'mixed race', 'minority'],
    'migrant_worker': ['migrant', 'migrant worker', 'foreign worker', 'bangladeshi', 'filipino', 'vietnamese', 'myanmar', 'burmese', 'prc', 'foreigner'],
    'religion': ['islam', 'muslim', 'christian', 'buddhist', 'hindu', 'taoist', 'sikh'],
}

SINGLE_RESPONSE_SCHEMA = {
    'type': 'object',
    'properties': {
        'text': {'type': 'string'},
        'label': {'type': 'string', 'enum': ['hate', 'offensive', 'normal']},
        'target_group': {'type': 'string'},
    },
    'required': ['text', 'label', 'target_group'],
    'additionalProperties': True,
}

def has_group_mention(text: str, target_group: str) -> bool:
    tg = (target_group or '').strip().lower()
    if tg in ('', 'none'):
        return True
    keywords = GROUP_KEYWORDS.get(tg, [tg.replace('_', ' ')])
    text_lower = (text or '').lower()
    return any(k in text_lower for k in keywords)

def _safe_preview(text, max_len=400):
    if not text:
        return ''
    text = str(text).replace('\n', '\\n')
    return text[:max_len] + ('...' if len(text) > max_len else '')

def _generate_single_json(prompt: str, temperature: float = 0.2, max_output_tokens: int = 320):
    response = GENAI_CLIENT.models.generate_content(
        model=GEMINI_MODEL,
        contents=prompt,
        config={
            'temperature': temperature,
            'top_p': 0.9,
            'max_output_tokens': max_output_tokens,
            'response_mime_type': 'application/json',
            'response_json_schema': SINGLE_RESPONSE_SCHEMA,
        },
    )

    finish_reasons = []
    for cand in getattr(response, 'candidates', []) or []:
        reason = getattr(cand, 'finish_reason', None)
        if reason is not None:
            finish_reasons.append(str(reason))

    raw = (getattr(response, 'text', None) or '').strip()
    return raw, {'finish_reasons': finish_reasons}

def _parse_single_json(raw_text: str):
    text = (raw_text or '').strip()
    if not text:
        return None
    text = re.sub(r'^```(?:json)?\s*', '', text)
    text = re.sub(r'\s*```$', '', text)
    try:
        data = json.loads(text)
    except Exception:
        return None

    if isinstance(data, list):
        if not data:
            return None
        data = data[0]
    if not isinstance(data, dict):
        return None

    if 'text' not in data or 'label' not in data:
        return None
    data['label'] = str(data.get('label', '')).strip().lower()
    data['target_group'] = str(data.get('target_group', 'none')).strip().lower()
    return data

def build_hate_prompt_for_group(term, context, target_group):
    group_desc = SG_TARGET_GROUPS.get(target_group, target_group)
    religion_note = ""
    if target_group == 'religion':
        religion_note = "For religion, explicitly mention one specific religion (Islam, Christianity, Buddhism, Hinduism, Taoism, or Sikhism) in the text."

    return f'''SYSTEM:\n{SYSTEM_PROMPT}\n\nGenerate exactly ONE JSON object for a hate example.

Term: \"{term}\"
Meaning/context:\n{context}

Required label: hate
Required target_group: {target_group}
Target definition: {group_desc}
{religion_note}

Hard rules:
- MUST contain the exact term \"{term}\" in the text
- MUST explicitly mention the target group in text
- English Singlish only, 1-3 sentences
- Output only JSON object with keys text,label,target_group
'''

def build_single_prompt(term, context, label):
    return f'''SYSTEM:\n{SYSTEM_PROMPT}\n\nGenerate exactly ONE JSON object.

Term: \"{term}\"
Meaning/context:\n{context}

Required label: {label}
For {label}, target_group must be \"none\".

Hard rules:
- MUST contain the exact term \"{term}\" in the text
- English Singlish only, 1-3 sentences
- Output only JSON object with keys text,label,target_group
'''

def generate_dataset(input_file, output_file, selected_terms, n_per_label=2, limit=None, temperature=0.2, max_retries=3):
    input_file = Path(input_file)
    output_file = Path(output_file)
    output_file.parent.mkdir(parents=True, exist_ok=True)
    failure_log_file = output_file.with_suffix(output_file.suffix + '.failures.log')

    entries = [normalize_entry(json.loads(line)) for line in input_file.open(encoding='utf-8') if line.strip()]
    entries = [e for e in entries if e['term']]

    if selected_terms:
        emap = {e['term'].casefold(): e for e in entries}
        entries = [emap[t.casefold()] for t in selected_terms if t.casefold() in emap]

    if limit:
        entries = entries[:limit]

    total_written = 0
    with output_file.open('a', encoding='utf-8') as fh, failure_log_file.open('a', encoding='utf-8') as flog:
        for idx, entry in enumerate(entries, start=1):
            term = entry['term']
            context = entry['context']
            if not context:
                print(f'[{idx}/{len(entries)}] {term} ... SKIP (no context)')
                continue

            groups = infer_target_groups(entry['languages'])
            groups = [g for g in groups if g in SG_TARGET_GROUPS]
            if not groups:
                groups = list(SG_TARGET_GROUPS.keys())

            term_rows = []
            term_failures = []

            hate_jobs = []
            for _ in range(n_per_label):
                for g in groups:
                    hate_jobs.append(('hate', g, build_hate_prompt_for_group(term, context, g)))

            for _ in range(n_per_label):
                hate_jobs.append(('offensive', 'none', build_single_prompt(term, context, 'offensive')))
                hate_jobs.append(('normal', 'none', build_single_prompt(term, context, 'normal')))

            total_subtasks = len(hate_jobs)
            print(f'[{idx}/{len(entries)}] {term} ... start ({total_subtasks} subtasks)', flush=True)
            term_start = time.time()

            for job_idx, (expected_label, expected_group, prompt) in enumerate(hate_jobs, start=1):
                success = False
                attempt_logs = []
                print(f'  [{job_idx}/{total_subtasks}] {expected_label}:{expected_group}', end=' ', flush=True)

                for attempt in range(1, max_retries + 1):
                    if attempt > 1:
                        print(f'(retry {attempt})', end=' ', flush=True)

                    raw, meta = _generate_single_json(prompt, temperature=temperature)
                    finish_reasons = (meta or {}).get('finish_reasons') or []
                    parsed = _parse_single_json(raw)

                    attempt_info = {
                        'attempt': attempt,
                        'finish_reasons': finish_reasons,
                        'raw_preview': _safe_preview(raw),
                        'reason': None,
                        'expected_label': expected_label,
                        'expected_target_group': expected_group,
                    }

                    if not parsed:
                        attempt_info['reason'] = 'invalid_or_non_json_output'
                        attempt_logs.append(attempt_info)
                        continue

                    text = parsed.get('text', '')
                    label = parsed.get('label', '').strip().lower()
                    tg = parsed.get('target_group', 'none').strip().lower()

                    if not term_appears_in_text(term, text):
                        attempt_info['reason'] = 'missing_source_term'
                        attempt_logs.append(attempt_info)
                        continue

                    if label != expected_label:
                        attempt_info['reason'] = f'label_mismatch:{label}'
                        attempt_logs.append(attempt_info)
                        continue

                    if expected_label == 'hate':
                        if tg != expected_group:
                            attempt_info['reason'] = f'target_group_mismatch:{tg}'
                            attempt_logs.append(attempt_info)
                            continue
                        if not has_group_mention(text, tg):
                            attempt_info['reason'] = 'hate_missing_target_group_mention'
                            attempt_logs.append(attempt_info)
                            continue
                    else:
                        parsed['target_group'] = 'none'

                    attempt_info['reason'] = 'success'
                    attempt_logs.append(attempt_info)

                    term_rows.append({
                        'text': parsed['text'],
                        'label': parsed['label'],
                        'target_group': parsed.get('target_group', 'none'),
                        'source_term': term,
                        'source_category': (entry['categories'][0] if entry['categories'] else ''),
                    })
                    success = True
                    print('✓', flush=True)
                    break

                if not success:
                    print('✗', flush=True)
                    term_failures.append({
                        'term': term,
                        'expected_label': expected_label,
                        'expected_target_group': expected_group,
                        'attempt_logs': attempt_logs,
                    })

            for row in term_rows:
                fh.write(json.dumps(row, ensure_ascii=False) + '\n')
                total_written += 1

            elapsed = time.time() - term_start
            if term_failures:
                for fail in term_failures:
                    fail['timestamp'] = time.time()
                    flog.write(json.dumps(fail, ensure_ascii=False) + '\n')
                if term_rows:
                    print(f'[{idx}/{len(entries)}] {term} ... PARTIAL ({len(term_rows)} rows, {len(term_failures)} failed subtasks, {elapsed:.1f}s)')
                else:
                    print(f'[{idx}/{len(entries)}] {term} ... FAILED ({elapsed:.1f}s)')
                    last = term_failures[-1]['attempt_logs'][-1] if term_failures[-1]['attempt_logs'] else {}
                    print('  fail_reason:', last.get('reason'))
                    print('  raw_preview:', (last.get('raw_preview') or '')[:180])
            else:
                print(f'[{idx}/{len(entries)}] {term} ... OK ({len(term_rows)} rows, {elapsed:.1f}s)')

            time.sleep(0.15)

    print('Done. Wrote', total_written, 'rows to', output_file)
    print('Failure log:', failure_log_file)

print('✅ Targeted per-group generation mode active (with live subtask progress)')

✅ Targeted per-group generation mode active (with live subtask progress)


## Generate dataset

In [ ]:
generate_dataset(
    input_file=INPUT_FILE,
    output_file=OUTPUT_FILE,
    selected_terms=SELECTED_TERMS,
    n_per_label=PER_TERM,
    limit=LIMIT,
    temperature=TEMPERATURE,
)

[1/11] anak babi ... start (48 subtasks)
  [1/48] hate:chinese 

✓
  [2/48] hate:malay (retry 2) (retry 3) ✓
  [3/48] hate:indian ✓
  [4/48] hate:others (retry 2) (retry 3) ✗
  [5/48] hate:migrant_worker (retry 2) (retry 3) ✗
  [6/48] hate:religion ✓
  [7/48] hate:chinese (retry 2) (retry 3) ✗
  [8/48] hate:malay (retry 2) (retry 3) ✗
  [9/48] hate:indian (retry 2) (retry 3) ✗
  [10/48] hate:others (retry 2) ✓
  [11/48] hate:migrant_worker (retry 2) (retry 3) ✗
  [12/48] hate:religion (retry 2) (retry 3) ✗
  [13/48] hate:chinese ✓
  [14/48] hate:malay ✓
  [15/48] hate:indian (retry 2) ✓
  [16/48] hate:others (retry 2) (retry 3) ✗
  [17/48] hate:migrant_worker (retry 2) (retry 3) ✗
  [18/48] hate:religion (retry 2) (retry 3) ✗
  [19/48] hate:chinese (retry 2) ✓
  [20/48] hate:malay ✓
  [21/48] hate:indian (retry 2) (retry 3) ✗
  [22/48] hate:others (retry 2) (retry 3) ✗
  [23/48] hate:migrant_worker (retry 2) (retry 3) ✗
  [24/48] hate:religion (retry 2) (retry 3) ✓
  [25/48] hate:chinese (retry 2) (retry 3) ✗
  [26/48] hate:malay (retry 2) (retry 3)

(retry 3) ✗
  [8/18] normal:none (retry 2) (retry 3) ✗
  [9/18] offensive:none 

: 

## Cleanup + sort existing output
Remove rows where `target_group` is set but not explicitly mentioned in text, then sort by `source_term`.

In [28]:
from pathlib import Path
import json
from collections import Counter

out_path = Path(OUTPUT_FILE)
if not out_path.exists():
    raise FileNotFoundError(f'Output file not found: {out_path}')

GROUP_KEYWORDS = {
    'chinese': ['chinese', 'hokkien', 'teochew', 'cantonese', 'mandarin'],
    'malay': ['malay', 'muslim', 'islam'],
    'indian': ['indian', 'tamil', 'hindi', 'punjabi', 'sikh'],
    'migrant_worker': ['migrant', 'migrant worker', 'foreign worker', 'bangladeshi', 'filipino', 'vietnamese', 'myanmar', 'burmese', 'prc', 'foreigner'],
    'religion': ['religion', 'muslim', 'islam', 'christian', 'buddhist', 'hindu', 'sikh'],
    'others': ['eurasian', 'minority', 'mixed-race', 'mixed race'],
}

def has_group_mention(text: str, target_group: str) -> bool:
    tg = (target_group or '').strip().lower()
    if tg in ('', 'none'):
        return True
    keywords = GROUP_KEYWORDS.get(tg, [tg.replace('_', ' ')])
    text_lower = (text or '').lower()
    return any(k in text_lower for k in keywords)

rows = [json.loads(line) for line in out_path.open(encoding='utf-8') if line.strip()]
before = len(rows)

kept = []
removed = []
for row in rows:
    tg = str(row.get('target_group', 'none')).strip().lower()
    text = row.get('text', '')
    if has_group_mention(text, tg):
        kept.append(row)
    else:
        row['_remove_reason'] = 'target_group_not_in_text'
        removed.append(row)

# Sort by source term (then label for stable grouping)
kept.sort(key=lambda r: ((r.get('source_term') or '').lower(), (r.get('label') or '').lower()))

# Backup original before overwrite
backup_path = out_path.with_suffix(out_path.suffix + '.backup_before_group_cleanup')
backup_path.write_text(''.join(json.dumps(r, ensure_ascii=False) + '\n' for r in rows), encoding='utf-8')

# Overwrite cleaned + sorted file
out_path.write_text(''.join(json.dumps(r, ensure_ascii=False) + '\n' for r in kept), encoding='utf-8')

print('Output file:', out_path)
print('Backup file:', backup_path)
print('Rows before:', before)
print('Rows removed:', len(removed))
print('Rows after:', len(kept))
print('Removed by target_group:', Counter((r.get('target_group') or 'none') for r in removed))

if removed:
    print('\nSample removed rows (up to 5):')
    for r in removed[:5]:
        print(f"- term={r.get('source_term')} | label={r.get('label')} | target_group={r.get('target_group')}")
        print(f"  text={r.get('text', '')[:180]}")

Output file: /content/drive/MyDrive/dap-hs/synthetic_training_gemini.jsonl
Backup file: /content/drive/MyDrive/dap-hs/synthetic_training_gemini.jsonl.backup_before_group_cleanup
Rows before: 90
Rows removed: 1
Rows after: 89
Removed by target_group: Counter({'others': 1})

Sample removed rows (up to 5):
- term=NBCB | label=hate | target_group=others
  text=These others and foreigners are stealing all our jobs, NBCB we should kick them out.


## Output check

In [29]:
from collections import Counter, defaultdict

out_file = Path(OUTPUT_FILE)
if not out_file.exists():
    raise FileNotFoundError(f'Output file not found: {out_file}')

rows = [json.loads(line) for line in out_file.open(encoding='utf-8') if line.strip()]
print('output file:', out_file)
print('rows:', len(rows))
print('unique terms:', len({r['source_term'] for r in rows}))
print('labels:', Counter(r['label'] for r in rows))

by_term = defaultdict(list)
for row in rows:
    by_term[row.get('source_term', 'UNKNOWN')].append(row)

available_terms = sorted(by_term.keys())
print('\nAvailable terms:')
for t in available_terms:
    print(f'- {t} ({len(by_term[t])} rows)')

TERM_TO_SHOW = None

def print_term_examples(term, examples):
    print('\n' + '=' * 100)
    print(f'TERM: {term} | rows: {len(examples)}')
    print('=' * 100)
    for idx, ex in enumerate(examples, start=1):
        print(f"[{idx}] label={ex.get('label')} | target_group={ex.get('target_group')}")
        print(ex.get('text', ''))
        print('-' * 100)

if TERM_TO_SHOW:
    if TERM_TO_SHOW not in by_term:
        raise ValueError(f"Term '{TERM_TO_SHOW}' not found. Choose from: {available_terms}")
    print_term_examples(TERM_TO_SHOW, by_term[TERM_TO_SHOW])
else:
    for term in available_terms:
        print_term_examples(term, by_term[term])

output file: /content/drive/MyDrive/dap-hs/synthetic_training_gemini.jsonl
rows: 89
unique terms: 5
labels: Counter({'normal': 30, 'offensive': 30, 'hate': 29})

Available terms:
- NBCB (17 rows)
- anak babi (18 rows)
- pariah (18 rows)
- siao lang (18 rows)
- tu lan (18 rows)

TERM: NBCB | rows: 17
[1] label=hate | target_group=chinese
Chinese people are so greedy and selfish, NBCB always taking everything for themselves.
----------------------------------------------------------------------------------------------------
[2] label=hate | target_group=malay
Malay people are so lazy and never want to work hard, NBCB always asking for handouts.
----------------------------------------------------------------------------------------------------
[3] label=hate | target_group=indian
Indian people are so loud and annoying on the bus, NBCB go back to your own country.
----------------------------------------------------------------------------------------------------
[4] label=hate | target_g